# Debug-notebook: Visualising FAISS-index

This notebook is only meant to debug and explore the FAISS-index

## What you can do here:
1. See index as table (faiss_id, vector, payload)
2. Statistics en verifications (chunk sizes, double chunks, vector length)
3. Plot vectors in 2D or 3D
4. Debug retrieval: which chunks are chosen for which question?

## How it works:
- Execute each cell from top to bottom
- If de index store changes (new PDF's, other chunking settings, other model ...), build it again using `python -m tests.try_index --rebuild` and start executing cells from "3. Loading Index" again

## 1. Setup

In [1]:
%load_ext autoreload
%autoreload 2

import sys  
from pathlib import Path 

ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))  

import numpy as np  
import pandas as pd  
import plotly.express as px  
from sklearn.decomposition import PCA  
from sklearn.manifold import TSNE  

from src.config import INDEX_DIR  
from src.embeddings import E5Embeddings  
from src.vectorstore import index_exists, load_index  

pd.set_option("display.max_colwidth", 80) 
pd.set_option("display.width", 200)  

if not index_exists():  
    raise FileNotFoundError("Geen index gevonden. Bouw hem eerst met: python -m tests.try_index")

## 2. Help functions

In [ ]:
def get_vectors(store):
    return store.index.reconstruct_n(0, store.index.ntotal)


def page_label(meta):
    page, page_end = meta.get("page"), meta.get("page_end")
    return str(page) if page == page_end else f"{page}-{page_end}"


def index_to_dataframe(store, vectors):
    rows = []  
    for faiss_id in range(store.index.ntotal): 
        doc_id = store.index_to_docstore_id[faiss_id] 
        chunk = store.docstore.search(doc_id)  
        meta = chunk.metadata
        rows.append({
            "faiss_id": faiss_id,  
            "source": meta.get("source"),  
            "pages": page_label(meta),  
            "chunk_index": meta.get("chunk_index"),  
            "chars": len(chunk.page_content),  
            "norm": float(np.linalg.norm(vectors[faiss_id])),  
            "text": chunk.page_content,  
            "vector": vectors[faiss_id],  
        })
    return pd.DataFrame(rows)


def show_chunk(faiss_id):
    row = df.loc[faiss_id]
    print(f"faiss_id {faiss_id} | {row['source']} | pagina {row['pages']} | {row['chars']} tekens | norm {row['norm']:.3f}")
    print("Eerste 10 getallen van de vector:", np.round(row["vector"][:10], 3))
    print("-" * 60)
    print(row["text"])


def top_k_table(question, k=5):
    results = store.similarity_search_with_score(question, k=k)  
    return pd.DataFrame([
        {
            "rank": rank,
            "score": round(float(score), 3),  
            "source": chunk.metadata.get("source"),
            "pages": page_label(chunk.metadata),
            "chunk_index": chunk.metadata.get("chunk_index"),
            "text": chunk.page_content[:150].replace("\n", " "),  
        }
        for rank, (chunk, score) in enumerate(results, start=1)  
    ])


def plot(method="pca", dims=2, query=None, k=3):
    plot_df = df[["faiss_id", "source", "pages"]].copy() 
    plot_df["label"] = df["text"].str.replace("\n", " ").str[:100]  
    plot_df["type"] = "chunk" 
    plot_df["size"] = 4  
    points = vectors  

    if query:
        query_vector = np.array(embeddings.embed_query(query))  
        scores = vectors @ query_vector  
        for i in np.argsort(scores)[::-1][:k]:  
            plot_df.loc[i, "type"] = f"top-{k}" 
            plot_df.loc[i, "size"] = 8
        query_row = pd.DataFrame([{"faiss_id": -1, "source": "VRAAG", "pages": "", "label": query, "type": "vraag", "size": 14}])
        plot_df = pd.concat([plot_df, query_row], ignore_index=True)  
        points = np.vstack([vectors, query_vector])  

    if method == "pca":
        pca = PCA(n_components=dims).fit(vectors)
        coords = pca.transform(points)  
        print(f"PCA behoudt {pca.explained_variance_ratio_.sum():.0%} van de variatie in {dims}D")
    else:
        coords = TSNE(
            n_components=dims,
            metric="cosine", 
            init="pca",  
            random_state=42,  
            perplexity=min(30, len(points) - 1),  
        ).fit_transform(points)  

    plot_df["x"], plot_df["y"] = coords[:, 0], coords[:, 1]
    hover = {"pages": True, "faiss_id": True, "type": True, "x": False, "y": False, "size": False}
    title = f"{method.upper()} {dims}D | {len(df)} chunks" + (f" | vraag: {query}" if query else "")

    if dims == 2:
        return px.scatter(plot_df, x="x", y="y", color="source", symbol="type", size="size", size_max=12,
                          hover_name="label", hover_data=hover, title=title)
    plot_df["z"] = coords[:, 2]  
    hover["z"] = False
    return px.scatter_3d(plot_df, x="x", y="y", z="z", color="source", symbol="type", size="size", size_max=12,
                         hover_name="label", hover_data=hover, title=title)

## 3. Loading Index


In [3]:
embeddings = E5Embeddings()  
store = load_index(embeddings) 
vectors = get_vectors(store)  
df = index_to_dataframe(store, vectors) 
print(f"{len(df)} chunks, vectordimensie {vectors.shape[1]}")

d:\School\Vives\Jaar 4\Trends in Business en IT\advanced-rag-chatbot\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2922.98it/s]


956 chunks, vectordimensie 384


## 4. Table: index, vector and payload

In [21]:
df.drop(columns=["norm", "chunk_index"]).loc[700:720] 

,faiss_id,source,pages,chars,text,vector
700,700,syllabus_ethiek_en_AI.pdf,34,868,Voorbeeld 9 (ter illustratie): New Atlantis\nIn zijn boek New Atlantis toont...,"[0.087813, -0.036775, -0.079829, -0.049742, 0.101935, -0.065576, 0.024499, 0..."
701,701,syllabus_ethiek_en_AI.pdf,34,453,D. Naar vandaag toe\nDe ontwikkeling van techniek lijkt vandaag geen grenzen...,"[0.035824, -0.07289, -0.081242, -0.061627, 0.054873, -0.063304, -0.000625, 0..."
702,702,syllabus_ethiek_en_AI.pdf,35,979,Verantwoordelijkheid\nEthiek en AI\n35\nTechniek heeft een eigen plaats vero...,"[0.09106, -0.020587, -0.062111, -0.061603, 0.054648, -0.036732, 0.015797, 0...."
703,703,syllabus_ethiek_en_AI.pdf,35,963,begrip technologie uit lijken te kunnen. Op een eerder negatieve manier beke...,"[0.067297, -0.034092, -0.084993, -0.069105, 0.070948, -0.047618, 0.038472, 0..."
704,704,syllabus_ethiek_en_AI.pdf,35,898,"We kopen bijvoorbeeld een cd-speler omwille van de muziek, of we kopen een\n...","[0.112803, -0.025647, -0.058385, -0.05113, 0.086354, -0.04768, 0.024295, 0.0..."
705,705,syllabus_ethiek_en_AI.pdf,35,290,"39 P. TIJMES, Albert Borgmann. Technologie en het karakter van het hedendaag...","[0.074751, -0.01563, -0.100437, -0.084016, 0.077899, -0.053964, 0.012229, 0...."
706,706,syllabus_ethiek_en_AI.pdf,36,982,Verantwoordelijkheid\nEthiek en AI\n36\nklankinstallatie die de genoemde CD ...,"[0.074367, -0.009287, -0.068237, -0.055993, 0.046823, -0.043087, 0.031566, 0..."
707,707,syllabus_ethiek_en_AI.pdf,36,819,staan van het product: deze vormen als het ware de verborgen machinerie.”40 ...,"[0.091164, -0.059291, -0.083024, -0.03761, 0.075031, -0.02059, 0.037091, 0.0..."
708,708,syllabus_ethiek_en_AI.pdf,36,959,Voorbeeld 10: Water\n“Het schone water stroomt overal en altijd gemakkelijk ...,"[0.039081, -0.07623, -0.084496, -0.050363, 0.065683, -0.040947, 0.008038, 0...."
709,709,syllabus_ethiek_en_AI.pdf,36-37,375,"40 P. TIJMES, Art. cit., p. 119.\nDe mechaniek is\nverborgen\n\nVerantwoorde...","[0.084911, 0.001621, -0.077922, -0.070595, 0.048786, -0.048217, -0.024, 0.04..."


Show 1 entire chunk (you can change the number to check a specific chunk):

In [20]:
show_chunk(700)

faiss_id 700 | syllabus_ethiek_en_AI.pdf | pagina 34 | 868 tekens | norm 1.000
Eerste 10 getallen van de vector: [ 0.088 -0.037 -0.08  -0.05   0.102 -0.066  0.024  0.04   0.009  0.027]
------------------------------------------------------------
Voorbeeld 9 (ter illustratie): New Atlantis
In zijn boek New Atlantis toont Bacon wat hij bedoelt met de stelling dat wetenschap
en techniek de maatschappij ten goede komen. In zijn boek beschrijft Bacon een
eiland, Bensalem genaamd, waarvan hij niet alleen stap voor stap de gebruiken, de
gewoontes en de gebruikers de revue laat passeren. Belangrijk voor onze thematiek
is dat er op het eiland een wetenschappelijk instituut wordt gesponserd door de
staat, dat Salomonshuis heet. In dat instituut worden experimenten gedaan volgens
de methode die Bacon heeft beschreven in zijn wetenschappelijk werk. Via de
experimenten slaagt het Salomonshuis erin om meer inzicht te krijgen in de
samenleving. Via deze nieuwe inzichten slaagt ze erin om de samenlevi

## 5. Statistics and verification

In [6]:
print("Dubbele chunks:", df["text"].duplicated().sum())  
print("\nLengte van de vectoren (verwacht: allemaal ±1.0 door de normalisatie):")
print(df["norm"].describe().round(4))

#Per PDF: amount of chunks and lengths
df.groupby("source").agg(
    chunks=("faiss_id", "count"),
    gem_lengte=("chars", "mean"),
    min_lengte=("chars", "min"),
    max_lengte=("chars", "max"),
).round(0)

Dubbele chunks: 1

Lengte van de vectoren (verwacht: allemaal ±1.0 door de normalisatie):
count    956.0
mean       1.0
std        0.0
min        1.0
25%        1.0
50%        1.0
75%        1.0
max        1.0
Name: norm, dtype: float64


,chunks,gem_lengte,min_lengte,max_lengte
source,,,,
OER2026-27.pdf,274,745.0,37,998
handleiding_polar.pdf,317,780.0,44,999
syllabus_ethiek_en_AI.pdf,365,720.0,44,998


In [7]:
px.histogram(df, x="chars", color="source", nbins=30, title="Chunklengte in tekens")

## 6. Plotting

A vector of 384 numbers can't be drawn, so we reduce it to 2 or 3 dimensions, a simplification.

- PCA is linear and fast. The displayed percentage indicates how much information the graph retains. If it is low, the graph may not accurately represent the actual distances.
- t-SNE keeps nearby points together. Focus only on which points are close to each other; distances between clusters are not meaningful.
- Color = PDF. Hover over a point to see its text, page number and faiss_id.

In [8]:
plot("pca", 2)

PCA behoudt 20% van de variatie in 2D


In [9]:
plot("tsne", 2)

In [10]:
plot("tsne", 3)  #3D

## 7. Debug Retrieval

Change the question and rerun both cells. The table displays the top-k results with their scores, while the graph shows the positions of the query (large dot) and the best matching chunks (highlighted).

In [25]:
question = "Wanneer wordt de examenplanning meegedeeld?"  #change to your own question about the PDF's
top_k_table(question, k=5)

,rank,score,source,pages,chunk_index,text
0,1,0.890,OER2026-27.pdf,41,116,Art. 50 Examenplanning De examenplanning wordt ten minste vier weken vooraf...
1,2,0.881,OER2026-27.pdf,39-40,112,2.2. Organisatie van de examens Art. 44 Examenperiodes Binnen een examenpe...
2,3,0.873,OER2026-27.pdf,53,149,Onderwijs- en Examenreglement 2025-2026 52 Art. 73 Schriftelijke examens Ee...
3,4,0.870,OER2026-27.pdf,67,193,De voorzitter van de examencommissie bepaalt het tijdstip en de wijze waarop...
4,5,0.865,OER2026-27.pdf,46,130,Onderwijs- en Examenreglement 2025-2026 45 spreidingsmogelijkheden worden va...


In [26]:
plot("tsne", 2, query=question, k=3)

## 8. Export

In [13]:
export = df.drop(columns="vector").assign(
    vector=df["vector"].apply(lambda v: " ".join(f"{x:.4f}" for x in v))  
)

path = INDEX_DIR / "index_table.csv"
export.to_csv(path, index=False, sep=";", encoding="utf-8-sig") 
print(f"Opgeslagen: {path}")

Opgeslagen: D:\School\Vives\Jaar 4\Trends in Business en IT\advanced-rag-chatbot\data\index\index_table.csv
